# Surface Detection Workflow

This notebook isolates the machined metal face from each image and lets you preview the mask before batch processing the full dataset.

## Environment

Use the project virtual environment as the notebook kernel when possible:

`source .venv/bin/activate`

If notebook packages are not installed yet, run:

`pip install -r requirements.txt`

In [3]:
!pip install tqdm

In [7]:
import cv2
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm

In [8]:
def detect_steel_rectangle(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Heavy blur to remove texture
    blur = cv2.GaussianBlur(gray, (21, 21), 0)

    # Threshold
    _, thresh = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Ensure object is white
    if np.mean(thresh) > 127:
        thresh = cv2.bitwise_not(thresh)

    # Morphological cleanup
    kernel = np.ones((15, 15), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    # Find contours
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    best_rect = None
    max_area = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 5000:
            continue

        rect = cv2.minAreaRect(cnt)
        box = cv2.boxPoints(rect)
        box = np.int32(box)

        if area > max_area:
            max_area = area
            best_rect = box

    return best_rect, thresh

In [9]:
detect_steel_rectangle('Data/16/16_4.jpeg') 

error: OpenCV(4.9.0) :-1: error: (-5:Bad argument) in function 'cvtColor'
> Overload resolution failed:
>  - src is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'src'


In [12]:
INPUT_ROOT = "Data"
OUTPUT_ROOT = "Processed"

for category in ["success", "failed", "debug"]:
    for i in range(1, 25):
        os.makedirs(f"{OUTPUT_ROOT}/{category}/{i}", exist_ok=True)

print("Folders ready ✅")

Folders ready ✅


In [13]:
def process_all():
    for folder_num in range(1, 25):
        folder_path = Path(INPUT_ROOT) / str(folder_num)

        if not folder_path.exists():
            continue

        print(f"\nProcessing Folder {folder_num}...")

        for img_path in tqdm(list(folder_path.glob("*"))):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            result_img = img.copy()

            rect, thresh = detect_steel_rectangle(img)

            # Draw if detected
            if rect is not None:
                cv2.drawContours(result_img, [rect], 0, (0, 255, 0), 3)

                save_path = Path(OUTPUT_ROOT) / "success" / str(folder_num) / img_path.name
                cv2.imwrite(str(save_path), result_img)

            else:
                save_path = Path(OUTPUT_ROOT) / "failed" / str(folder_num) / img_path.name
                cv2.imwrite(str(save_path), result_img)

            # Always save debug
            debug_path = Path(OUTPUT_ROOT) / "debug" / str(folder_num) / img_path.name
            cv2.imwrite(str(debug_path), thresh)

In [17]:
process_all()


Processing Folder 1...


100%|██████████| 7/7 [00:00<00:00, 13.58it/s]



Processing Folder 2...


100%|██████████| 6/6 [00:00<00:00, 12.60it/s]



Processing Folder 3...


100%|██████████| 6/6 [00:00<00:00, 12.25it/s]


... (38 outputs left out)


Processing Folder 23...


100%|██████████| 6/6 [00:00<00:00, 13.24it/s]



Processing Folder 24...


100%|██████████| 6/6 [00:00<00:00, 13.05it/s]


In [10]:

def test_single(image_path):
    img = cv2.imread(image_path)

    rect, thresh = detect_steel_rectangle(img)

    vis = img.copy()

    if rect is not None:
        cv2.drawContours(vis, [rect], 0, (0,255,0), 3)
        print("✅ Detected")
    else:
        print("❌ Failed")

    cv2.imshow("Original", img)
    cv2.imshow("Threshold", thresh)
    cv2.imshow("Result", vis)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [ ]:
test_single("Data/9/9_1.jpeg")

✅ Detected
